# ITEM 1 & 1A EXTRACTION FROM 10-K FILINGS

**Purpose:** Extract Business (Item 1) and Risk Factors (Item 1A) sections from 10-K filings

**What This Does:**
- Extracts Item 1 (Business Description) and Item 1A (Risk Factors)
- Creates separate JSON files from MD&A extractions
- Organizes by year subfolders (2010-2025)
- Generates analysis-ready CSV and Parquet files

**Prerequisites:**
- Repository: `/content/drive/MyDrive/EDGAR_Project/edgar-crawler`
- Raw 10-K files already downloaded in `datasets/RAW_FILINGS/10-K/`
- Metadata file: `datasets/FILINGS_METADATA.csv`
- Year filter: 2010 onwards (~77,000 filings)

**Output Structure:**
```
datasets/EXTRACTED_FILINGS/
├── 10-K/              # Existing MD&A (Item 7) - UNTOUCHED
└── item_1_1a/         # NEW - Items 1 & 1A
    ├── 2010/
    ├── 2011/
    └── ...
```

---


## SECTION 1: SETUP (Run Every Time)

Run these cells at the start of every Colab session


In [ ]:
## 🟢 Cell 1: Mount Google Drive
import os
from google.colab import drive

if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive already mounted")
else:
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")


In [ ]:
## 🟢 Cell 2: Navigate to Repository
import os

REPO_DIR = '/content/drive/MyDrive/EDGAR_Project/edgar-crawler'

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Working directory: {os.getcwd()}")
else:
    print(f"❌ Repository not found at: {REPO_DIR}")


In [ ]:
## Cell 3: Install Dependencies
print("Installing dependencies...")

!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow

print("Dependencies installed")

In [ ]:
## Cell 4: Keep-Alive Script
from IPython.display import display, Javascript

display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))

print("Keep-alive activated")

---

## SECTION 2: CONFIGURATION

Create extraction configuration for Items 1 and 1A

In [ ]:
## Cell 5: Create Extraction Config
import json
import os

config_dir = 'extraction_configs'
os.makedirs(config_dir, exist_ok=True)

config_path = os.path.join(config_dir, 'items_1_1a.json')

config = {
    "description": "Extract Business (Item 1) and Risk Factors (Item 1A) from 10-K filings",
    "items_to_extract": ["1", "1A"],
    "filing_types": ["10-K"],
    "output_dir": "item_1_1a",
    "remove_tables": True,
    "skip_existing": True,
    "include_signature": False
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("Configuration created successfully!")
print(f"Config file: {config_path}")
print(json.dumps(config, indent=2))

---

## SECTION 3: FILTER METADATA

Load and filter metadata to only process filings from 2010 onwards

In [ ]:
## Cell 6: Load and Filter Metadata
import pandas as pd

metadata_path = 'datasets/FILINGS_METADATA.csv'

print("Loading metadata...")

if not os.path.exists(metadata_path):
    print(f"Metadata file not found: {metadata_path}")
else:
    metadata = pd.read_csv(metadata_path)
    print(f"Loaded {len(metadata):,} total filings")
    
    # Filter to 10-K and 2010 onwards
    metadata_10k = metadata[metadata['Type'] == '10-K'].copy()
    metadata_filtered = metadata_10k[metadata_10k['year'] >= 2010].copy()
    
    print(f"\nFiltered Metadata:")
    print(f"   Filings >= 2010: {len(metadata_filtered):,}")
    print(f"   Excluded (before 2010): {len(metadata_10k) - len(metadata_filtered):,}")
    
    # Year distribution
    print(f"\nYear Distribution:")
    year_counts = metadata_filtered['year'].value_counts().sort_index()
    for year, count in year_counts.items():
        print(f"      {year}: {count:,} filings")
    
    # Save filtered metadata
    filtered_path = 'datasets/FILINGS_METADATA_2010_onwards.csv'
    metadata_filtered.to_csv(filtered_path, index=False)
    print(f"\nFiltered metadata saved: {filtered_path}")

---

## SECTION 4: CREATE OUTPUT DIRECTORY STRUCTURE

Create the base directory (year subfolders created automatically during extraction)

In [ ]:
## Cell 7: Create Output Base Directory
import os

output_base = 'datasets/EXTRACTED_FILINGS/item_1_1a'
os.makedirs(output_base, exist_ok=True)

print(f"Base directory created: {output_base}/")
print("(Year subfolders will be created automatically during extraction)")

---

## SECTION 5: RUN EXTRACTION

Extract Items 1 and 1A from 10-K filings (2010 onwards)

In [ ]:
## Cell 8: Update config.json for Items 1 & 1A Extraction
import json

# Load main config
with open('config.json', 'r') as f:
    config = json.load(f)

# Update extract_items section for Items 1 & 1A
config['extract_items']['filings_metadata_file'] = 'FILINGS_METADATA_2010_onwards.csv'
config['extract_items']['filing_types'] = ['10-K']
config['extract_items']['items_to_extract'] = ['1', '1A']
config['extract_items']['remove_tables'] = True
config['extract_items']['skip_extracted_filings'] = True

# Save updated config
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Main config.json updated for Items 1 & 1A extraction")
print("Items to extract: 1 (Business), 1A (Risk Factors)")

In [ ]:
## Cell 9: Run Extraction
print("="*70)
print(" STARTING EXTRACTION: ITEMS 1 & 1A")
print("="*70)
print("\nConfiguration:")
print("   Items: 1 (Business), 1A (Risk Factors)")
print("   Output: datasets/EXTRACTED_FILINGS/10-K/ (will reorganize after)")
print("   Years: 2010-2025")
print("\nEstimated time: 8-12 hours\n")
print("="*70)

!python flexible_extractor.py --config extraction_configs/items_1_1a.json

### Reorganize Extracted Files

**Important:** Run this cell after extraction completes to move files from `10-K/` to `item_1_1a/`

In [ ]:
## Cell 9.5: Reorganize Files to item_1_1a Directory
import os
import shutil
import json
from tqdm import tqdm

print("Reorganizing extracted files to item_1_1a directory...\n")

source_base = 'datasets/EXTRACTED_FILINGS/10-K'
dest_base = 'datasets/EXTRACTED_FILINGS/item_1_1a'

# Create destination base
os.makedirs(dest_base, exist_ok=True)

moved_count = 0
skipped_count = 0

# Walk through year subfolders in source
for year_folder in sorted(os.listdir(source_base)):
    year_path = os.path.join(source_base, year_folder)
    
    if not os.path.isdir(year_path):
        continue
    
    # Only process years >= 2010
    try:
        if int(year_folder) < 2010:
            continue
    except:
        continue
    
    # Create corresponding year folder in destination
    dest_year_path = os.path.join(dest_base, year_folder)
    os.makedirs(dest_year_path, exist_ok=True)
    
    # Move JSON files that have Items 1 & 1A
    json_files = [f for f in os.listdir(year_path) if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Year {year_folder}", leave=False):
        source_file = os.path.join(year_path, filename)
        dest_file = os.path.join(dest_year_path, filename)
        
        # Check if already moved
        if os.path.exists(dest_file):
            skipped_count += 1
            continue
        
        # Check if this file has Items 1 or 1A
        try:
            with open(source_file, 'r') as f:
                data = json.load(f)
            
            # Only move if it has Items 1 or 1A
            has_item_1 = 'item_1' in data and len(data.get('item_1', '')) > 0
            has_item_1a = 'item_1a' in data and len(data.get('item_1a', '')) > 0
            
            if has_item_1 or has_item_1a:
                shutil.move(source_file, dest_file)
                moved_count += 1
        except Exception as e:
            print(f"\nWarning: Error processing {filename}: {e}")

print(f"\nReorganization complete!")
print(f"   Moved: {moved_count:,} files to {dest_base}/")
print(f"   Skipped (already exist): {skipped_count:,} files")

---

## SECTION 6: CHECK PROGRESS

Monitor extraction progress and verify output quality

In [ ]:
## Cell 10: Check Extraction Progress
import os
import json
import pandas as pd
from collections import defaultdict

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'

print("Scanning for extracted files...\n")

# Count files by year
year_counts = defaultdict(int)
all_files = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    if json_files and root != extracted_dir:
        year = os.path.basename(root)
        year_counts[year] = len(json_files)
        all_files.extend([os.path.join(root, f) for f in json_files])

# Load expected count
if os.path.exists('datasets/FILINGS_METADATA_2010_onwards.csv'):
    metadata = pd.read_csv('datasets/FILINGS_METADATA_2010_onwards.csv')
    expected = len(metadata)
else:
    expected = 77000  # Approximate

print("="*70)
print(" EXTRACTION PROGRESS")
print("="*70)
print(f"\nOverall Progress:")
print(f"   Extracted: {len(all_files):,} filings")
print(f"   Expected: {expected:,} filings")
print(f"   Progress: {len(all_files)/expected*100:.1f}%")
print(f"   Remaining: {max(0, expected - len(all_files)):,} filings")

# Display by year
if year_counts:
    print(f"\nFiles by Year:")
    for year in sorted(year_counts.keys()):
        print(f"      {year}: {year_counts[year]:,} files")

# Quality check
if len(all_files) > 0:
    print(f"\nQuality Check (5 random samples):\n")
    import random
    samples = random.sample(all_files, min(5, len(all_files)))
    
    for fpath in samples:
        fname = os.path.basename(fpath)
        try:
            with open(fpath, 'r') as f:
                data = json.load(f)
            
            item1_len = len(data.get('item_1', ''))
            item1a_len = len(data.get('item_1a', ''))
            
            status1 = 'YES' if item1_len > 100 else 'NO'
            status1a = 'YES' if item1a_len > 100 else 'NO'
            
            print(f"   {fname[:45]:45s}")
            print(f"      Item 1:  {status1:3s} ({item1_len:>7,} chars)")
            print(f"      Item 1A: {status1a:3s} ({item1a_len:>7,} chars)\n")
        except Exception as e:
            print(f"   {fname}: Error - {e}\n")

print("="*70)

if len(all_files) >= expected * 0.95:  # At least 95% complete
    print("\nEXTRACTION COMPLETE!")
    print("\n   Next: Run SECTION 7 to create analysis files")
else:
    print(f"\nExtraction in progress...")
    print(f"   {max(0, expected - len(all_files)):,} files remaining")

---

## SECTION 7: CREATE ANALYSIS FILES

Generate metadata CSV and consolidated Parquet file for analysis

In [ ]:
## Cell 11: Create Metadata CSV
import os
import json
import pandas as pd
from tqdm import tqdm

print("Creating metadata CSV for Items 1 & 1A...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
metadata_records = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            metadata_records.append({
                'filename': filename,
                'cik': filing.get('cik', ''),
                'company': filing.get('company', ''),
                'filing_date': filing.get('filing_date', ''),
                'year': filing.get('period_of_report', '')[:4] if filing.get('period_of_report') else '',
                'has_item_1': 'item_1' in filing and len(filing.get('item_1', '')) > 0,
                'item_1_length': len(filing.get('item_1', '')),
                'has_item_1a': 'item_1a' in filing and len(filing.get('item_1a', '')) > 0,
                'item_1a_length': len(filing.get('item_1a', '')),
                'json_path': filepath
            })
        except Exception as e:
            print(f"Warning: Error processing {filename}")

# Create DataFrame and save
df_meta = pd.DataFrame(metadata_records)
meta_path = 'datasets/items_1_1a_metadata.csv'
df_meta.to_csv(meta_path, index=False)

print(f"\nMetadata CSV created!")
print(f"   Location: {meta_path}")
print(f"   Records: {len(df_meta):,}")
print(f"   File size: {os.path.getsize(meta_path) / 1024:.1f} KB")

print(f"\nSummary Statistics:")
print(f"   Total filings: {len(df_meta):,}")
print(f"   Filings with Item 1: {df_meta['has_item_1'].sum():,}")
print(f"   Filings with Item 1A: {df_meta['has_item_1a'].sum():,}")
print(f"   Filings with BOTH: {(df_meta['has_item_1'] & df_meta['has_item_1a']).sum():,}")

print(f"\nLength Statistics:")
print(f"   Item 1 avg: {df_meta['item_1_length'].mean():,.0f} characters")
print(f"   Item 1A avg: {df_meta['item_1a_length'].mean():,.0f} characters")

print(f"\nSample Records:")
print(df_meta[['company', 'year', 'has_item_1', 'has_item_1a', 'item_1_length', 'item_1a_length']].head(10).to_string(index=False))

In [ ]:
## Cell 12: Create Consolidated Parquet File
import os
import json
import pandas as pd
from tqdm import tqdm

print("Creating consolidated Parquet file...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
full_data = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            if ('item_1' in filing and len(filing.get('item_1', '')) > 0) or \
               ('item_1a' in filing and len(filing.get('item_1a', '')) > 0):
                full_data.append({
                    'cik': filing.get('cik', ''),
                    'company': filing.get('company', ''),
                    'filing_date': filing.get('filing_date', ''),
                    'year': filing.get('period_of_report', '')[:4] if filing.get('period_of_report') else '',
                    'item_1_text': filing.get('item_1', ''),
                    'item_1a_text': filing.get('item_1a', '')
                })
        except Exception as e:
            pass

# Create DataFrame and save
df_full = pd.DataFrame(full_data)
parquet_path = 'datasets/items_1_1a_full.parquet'
df_full.to_parquet(parquet_path, compression='gzip', index=False)

print(f"\nParquet file created!")
print(f"   Location: {parquet_path}")
print(f"   Records: {len(df_full):,}")
print(f"   File size: {os.path.getsize(parquet_path) / (1024**2):.1f} MB")

print(f"\nText Length Statistics:")
print(f"\n   Item 1 (Business):")
print(df_full['item_1_text'].str.len().describe().to_string())

print(f"\n   Item 1A (Risk Factors):")
print(df_full['item_1a_text'].str.len().describe().to_string())

print(f"\nYear Coverage:")
print(f"   Years: {sorted(df_full['year'].unique())}")
print(f"   Range: {df_full['year'].min()} - {df_full['year'].max()}")

---

## SECTION 8: RESUME HELPER

Quick cells to resume extraction after disconnection

In [ ]:
## Cell 13: Quick Resume (Run After Disconnection)
from google.colab import drive
from IPython.display import display, Javascript
import os

print("Quick Resume After Disconnection\n")

# 1. Remount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print("Drive mounted")

# 2. Navigate to repo
os.chdir('/content/drive/MyDrive/EDGAR_Project/edgar-crawler')
print(f"Working directory: {os.getcwd()}")

# 3. Install dependencies
print("\nInstalling dependencies...")
!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow
print("Dependencies installed")

# 4. Keep-alive
display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))
print("Keep-alive activated")

print("\nReady to resume!")

In [ ]:
## Cell 14: Resume Extraction
print("Resuming extraction...\n")
!python flexible_extractor.py --config extraction_configs/items_1_1a.json

---

## EXTRACTION COMPLETE!

### Summary of Outputs:

1. **JSON Files** (organized by year):
   - Location: `datasets/EXTRACTED_FILINGS/item_1_1a/`
   - Structure: Year subfolders (2010-2025)
   - Each file contains: `item_1` (Business) and `item_1a` (Risk Factors)

2. **Metadata CSV**:
   - Location: `datasets/items_1_1a_metadata.csv`
   - Contains: File info, lengths, flags for Items 1 & 1A

3. **Parquet File** (compressed):
   - Location: `datasets/items_1_1a_full.parquet`
   - Contains: Full text for Items 1 & 1A
   - Use for: Text analysis, NLP, machine learning

### Next Steps:

- **Perform text analysis** (sentiment, readability, etc.)
- **Compare with MD&A data** (Item 7)
- **Research applications**: Risk analysis, business model comparison

---

**Questions or Issues?**
- Check the repository: https://github.com/haowenluo/edgar-crawler

---